# 01 · Limpeza dos dados

**Projeto Final · Visualização de Dados · ** — Etapa 2: Preparação dos dados

Este notebook prepara os dois datasets do projeto e responde às perguntas da Etapa 1 só com números:

Cada bloco de código vem depois de uma célula de texto que explica **o que** foi feito e **por quê**.

### Perguntas da Etapa 1

**Músicas (Spotify)**
1. Quais gêneros de playlist têm as músicas mais populares?
2. A popularidade das músicas muda conforme o ano de lançamento?
3. Características do áudio (energia, acusticidade, fala, ao vivo) têm relação com a popularidade?
4. As músicas ficaram mais curtas ao longo das décadas?

**Filmes (TMDB)**

5. Quais gêneros de filme têm as maiores notas médias?
6. Como a quantidade de filmes e a nota média mudam por década de lançamento?
7. Filmes com mais votos também têm notas mais altas?
8. Quais idiomas originais aparecem mais e como são avaliados?

In [61]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")
from src.utils import DATA, LIMPOS, GENEROS_MUSICA, IDIOMAS

pd.set_option("display.max_columns", 30)
LIMPOS.mkdir(exist_ok=True)

---
## Parte A · Músicas (Spotify)

### A1. Carregar
O dataset `spotify_songs.csv` lista músicas de playlists do Spotify, com popularidade (0 a 100), gênero da playlist e características de áudio calculadas pelo Spotify.

In [62]:
musicas = pd.read_csv(DATA / "musica" / "spotify_songs.csv")

### A2. Inspecionar
Antes de mexer, conferimos o tamanho, os tipos e os valores ausentes.

In [63]:
print("Linhas x colunas:", musicas.shape)
musicas.info()

Linhas x colunas: (32833, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32833 entries, 0 to 32832
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  32833 non-null  object 
 1   track_name                32828 non-null  object 
 2   track_artist              32828 non-null  object 
 3   track_popularity          32833 non-null  int64  
 4   track_album_id            32833 non-null  object 
 5   track_album_name          32828 non-null  object 
 6   track_album_release_date  32833 non-null  object 
 7   playlist_name             32833 non-null  object 
 8   playlist_id               32833 non-null  object 
 9   playlist_genre            32833 non-null  object 
 10  playlist_subgenre         32833 non-null  object 
 11  danceability              32833 non-null  float64
 12  energy                    32833 non-null  float64
 13  key                       32833

In [64]:
musicas.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.718,0.930,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.650,0.833,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [65]:
musicas.isna().sum()[lambda s: s > 0]

track_name          5
track_artist        5
track_album_name    5
dtype: int64

**O que a inspeção mostrou:**
- 32.833 linhas e 23 colunas.
- A data de lançamento do álbum (`track_album_release_date`) está como **texto** e com formatos diferentes: `2019-06-14`, `2019-06` ou só `2019`.
- A duração está em milissegundos, pouco legível.
- 5 músicas não têm nome, artista nem álbum.
- Uma mesma música (`track_id`) aparece em várias playlists, o que é esperado. Mas também há músicas repetidas **dentro da mesma playlist**, o que é duplicação de verdade (tratada no passo A6).

### A3. Selecionar e renomear
Ficamos só com as colunas que ajudam a responder às perguntas 1 a 4 e traduzimos os nomes para português (seguindo o tratamento que a equipe já tinha começado em `songs_tratamento.ipynb`).

Mantivemos `track_id` e `playlist_id` por enquanto, porque eles são necessários para encontrar duplicados. `playlist_id` é descartado depois do passo A6.

Removemos `danceability`, `key`, `loudness`, `mode`, `instrumentalness`, `valence` e `tempo`, que não entram nas perguntas.

In [66]:
colunas_descartadas = ["track_album_id", "danceability", "key", "loudness", "mode",
                       "instrumentalness", "valence", "tempo"]
musicas = musicas.drop(columns=colunas_descartadas)

musicas = musicas.rename(columns={
    "track_id": "id_musica",
    "track_name": "nome_musica",
    "track_artist": "artista",
    "track_popularity": "popularidade_musica",
    "track_album_name": "album_musica",
    "track_album_release_date": "data_lancamento_album",
    "playlist_name": "nome_playlist",
    "playlist_id": "id_playlist",
    "playlist_genre": "genero_playlist",
    "playlist_subgenre": "subgenero_playlist",
    "energy": "energia",
    "speechiness": "fala",
    "acousticness": "acusticidade",
    "liveness": "prob_aovivo",
    "duration_ms": "duracao_ms",
})
musicas.columns.tolist()

['id_musica',
 'nome_musica',
 'artista',
 'popularidade_musica',
 'album_musica',
 'data_lancamento_album',
 'nome_playlist',
 'id_playlist',
 'genero_playlist',
 'subgenero_playlist',
 'energia',
 'fala',
 'acusticidade',
 'prob_aovivo',
 'duracao_ms']

### A4. Corrigir tipos
- **Ano de lançamento:** a data vem em três formatos. O ano (4 primeiros caracteres) existe em todos, então extraímos `ano_lancamento` como inteiro e criamos `decada` para a pergunta 4. Não usamos a data completa porque 1.886 linhas só têm o ano.
- **Duração:** convertemos de milissegundos para minutos (`duracao_min`) e **removemos `duracao_ms`**. No notebook anterior, o `drop` não tinha sido atribuído de volta ao DataFrame, então a coluna continuava lá.

In [67]:
print(musicas["data_lancamento_album"].str.len().value_counts())

musicas["ano_lancamento"] = musicas["data_lancamento_album"].str[:4].astype(int)
musicas["decada"] = (musicas["ano_lancamento"] // 10) * 10

musicas["duracao_min"] = (musicas["duracao_ms"] / 60000).round(2)
musicas = musicas.drop(columns=["duracao_ms", "data_lancamento_album"])

musicas[["ano_lancamento", "decada", "duracao_min"]].describe()

data_lancamento_album
10    30947
4      1855
7        31
Name: count, dtype: int64


,ano_lancamento,decada,duracao_min
count,32833.000000,32833.000000,32833.000000
mean,2011.137270,2004.952639,3.763335
std,11.417451,10.666666,0.997240
min,1957.000000,1950.000000,0.070000
25%,2008.000000,2000.000000,3.130000
50%,2016.000000,2010.000000,3.600000
75%,2019.000000,2010.000000,4.230000
max,2020.000000,2020.000000,8.630000


### A5. Tratar ausentes
Apenas 5 linhas (0,02%) não têm nome, artista e álbum. Sem o nome não dá para identificar a música, e a perda é desprezível, então **removemos** essas linhas.

In [68]:
print("Antes:", len(musicas))
musicas = musicas.dropna(subset=["nome_musica", "artista"])
print("Depois:", len(musicas))
musicas.isna().sum().sum()

Antes: 32833
Depois: 32828


0

### A6. Remover duplicados
Há dois tipos de repetição, e tratamos cada um de forma diferente:

1. **A mesma música repetida na mesma playlist** (mesmo `id_musica` e `id_playlist`). Isso é erro de coleta e inflaria contagens e médias, então **removemos**.
2. **A mesma música em playlists diferentes.** Isso é informação real (a música está em várias playlists, às vezes de gêneros diferentes), então **mantemos**. Cada linha passa a significar "uma música dentro de uma playlist".

Para as perguntas que são sobre a música em si (2, 3 e 4), usamos a versão **sem repetição de `id_musica`** no passo de estatísticas, para não contar a mesma música duas vezes.

In [69]:
print("Duplicados exatos:", musicas.duplicated().sum())
print("Mesma música na mesma playlist:", musicas.duplicated(subset=["id_musica", "id_playlist"]).sum())

musicas = musicas.drop_duplicates(subset=["id_musica", "id_playlist"])
musicas = musicas.drop(columns=["id_playlist"])

print("Linhas após remover:", len(musicas))
print("Músicas únicas:", musicas["id_musica"].nunique())

Duplicados exatos: 0
Mesma música na mesma playlist: 582
Linhas após remover: 32246
Músicas únicas: 28352


### A7. Padronizar
- Removemos espaços extras dos textos.
- Trocamos os códigos de gênero (`r&b`, `latin`...) por nomes de exibição (`R&B`, `Latina`...), que vão direto para os gráficos.
- **Durações muito curtas:** há faixas com poucos segundos (vinhetas e intros). Removemos as que têm menos de 1 minuto, porque não são músicas completas e distorcem a pergunta 4.

In [70]:
for col in ["nome_musica", "artista", "album_musica", "nome_playlist", "subgenero_playlist"]:
    musicas[col] = musicas[col].str.strip()

musicas["genero_playlist"] = musicas["genero_playlist"].map(GENEROS_MUSICA)
print("Gêneros sem correspondência:", musicas["genero_playlist"].isna().sum())

curtas = musicas["duracao_min"] < 1
print("Faixas com menos de 1 minuto:", curtas.sum())
musicas = musicas[~curtas]
print("Linhas finais:", len(musicas))

Gêneros sem correspondência: 0
Faixas com menos de 1 minuto: 25
Linhas finais: 32221


### A8. Salvar

In [71]:
musicas.to_csv(LIMPOS / "musicas_limpas.csv", index=False)
musicas.head()

,id_musica,nome_musica,artista,popularidade_musica,album_musica,nome_playlist,genero_playlist,subgenero_playlist,energia,fala,acusticidade,prob_aovivo,ano_lancamento,decada,duracao_min
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,I Don't Care (with Justin Bieber) [Loud Luxury...,Pop Remix,Pop,dance pop,0.916,0.0583,0.1020,0.0653,2019,2010,3.25
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,Memories (Dillon Francis Remix),Pop Remix,Pop,dance pop,0.815,0.0373,0.0724,0.3570,2019,2010,2.71
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,All the Time (Don Diablo Remix),Pop Remix,Pop,dance pop,0.931,0.0742,0.0794,0.1100,2019,2010,2.94
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,Call You Mine - The Remixes,Pop Remix,Pop,dance pop,0.930,0.1020,0.0287,0.2040,2019,2010,2.82
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,Someone You Loved (Future Humans Remix),Pop Remix,Pop,dance pop,0.833,0.0359,0.0803,0.0833,2019,2010,3.15


---
## Parte B · Filmes (TMDB)

### B1. Carregar
O `Top Movies dataset.csv` traz quase 10 mil filmes populares do TMDB, com data de lançamento, popularidade, número de votos, nota média, idioma original e gêneros.

In [72]:
filmes = pd.read_csv(DATA / "filmes" / "Top Movies dataset.csv")

### B2. Inspecionar

In [73]:
print("Linhas x colunas:", filmes.shape)
filmes.info()

Linhas x colunas: (9837, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9837 entries, 0 to 9836
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Release_Date       9837 non-null   object 
 1   Title              9828 non-null   object 
 2   Overview           9828 non-null   object 
 3   Popularity         9827 non-null   float64
 4   Vote_Count         9827 non-null   object 
 5   Vote_Average       9827 non-null   object 
 6   Original_Language  9827 non-null   object 
 7   Genre              9826 non-null   object 
 8   Poster_Url         9826 non-null   object 
 9   Working URL        9837 non-null   object 
dtypes: float64(1), object(9)
memory usage: 768.6+ KB


In [74]:
filmes.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url,Working URL
0,15/12/2021,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...,https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,01/03/2022,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...,https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,25/02/2022,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,24/11/2021,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...,https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,22/12/2021,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


In [75]:
filmes.isna().sum()

Release_Date          0
Title                 9
Overview              9
Popularity           10
Vote_Count           10
Vote_Average         10
Original_Language    10
Genre                11
Poster_Url           11
Working URL           0
dtype: int64

In [76]:
# Linhas com problema: sem popularidade
filmes[filmes["Popularity"].isna()][["Release_Date", "Title", "Overview", "Working URL"]]

,Release_Date,Title,Overview,Working URL
1105,20/10/2013,Pixie Hollow Bake Off,Tink challenges Gelata to see who can bake the...,0
1106,- Just Desserts,NaN,NaN,0
1107,- If The Hue Fits,NaN,NaN,0
1108,- Dust Up,NaN,NaN,0
1109,- Scents And Sensibility,NaN,NaN,0
1110,- Just One Of The Girls,NaN,NaN,0
1111,- Volleybug,NaN,NaN,0
1112,- Hide And Tink,NaN,NaN,0
1113,- Rainbow's Ends,NaN,NaN,0
1114,- Fawn And Games,NaN,NaN,0


**O que a inspeção mostrou:**
- **Números guardados como texto:** `Vote_Count` e `Vote_Average` estão como `str`.
- **Data como texto** no formato brasileiro `dd/mm/aaaa`.
- **11 linhas quebradas.** A sinopse de *Pixie Hollow Bake Off* tem quebras de linha no CSV original, e cada pedaço virou uma "linha" nova (`- Just Desserts`, `- Dust Up`...). Essas linhas não têm nenhum dado numérico e vêm com `Working URL = 0`.
- `Genre` junta vários gêneros num texto só (`"Action, Adventure, Science Fiction"`).
- `Poster_Url`, `Working URL` e `Overview` não servem para as perguntas.

### B3. Selecionar e renomear
Descartamos as colunas de imagem e a sinopse, e traduzimos os nomes.

In [77]:
filmes = filmes.drop(columns=["Overview", "Poster_Url", "Working URL"])
filmes = filmes.rename(columns={
    "Release_Date": "data_lancamento",
    "Title": "titulo",
    "Popularity": "popularidade",
    "Vote_Count": "qtd_votos",
    "Vote_Average": "nota_media",
    "Original_Language": "idioma_original",
    "Genre": "generos",
})
filmes.columns.tolist()

['data_lancamento',
 'titulo',
 'popularidade',
 'qtd_votos',
 'nota_media',
 'idioma_original',
 'generos']

### B4. Corrigir tipos
- `qtd_votos` e `nota_media` viram números com `pd.to_numeric(errors="coerce")`. O que não for número vira `NaN` e é tratado no próximo passo.
- `data_lancamento` vira data com o formato explícito `%d/%m/%Y`. Sem o formato, o pandas poderia trocar dia e mês.
- Criamos `ano` e `decada` a partir da data.

In [78]:
filmes["qtd_votos"] = pd.to_numeric(filmes["qtd_votos"], errors="coerce")
filmes["nota_media"] = pd.to_numeric(filmes["nota_media"], errors="coerce")
filmes["data_lancamento"] = pd.to_datetime(filmes["data_lancamento"], format="%d/%m/%Y", errors="coerce")

filmes.dtypes

data_lancamento    datetime64[ns]
titulo                     object
popularidade              float64
qtd_votos                 float64
nota_media                float64
idioma_original            object
generos                    object
dtype: object

### B5. Tratar ausentes
1. **Linhas quebradas:** as 11 linhas sem popularidade são os pedaços da sinopse quebrada (e o filme a que ela pertence, que ficou sem nenhum dado numérico). Não há como recuperar os números, então **removemos**.
2. **Filmes com 0 votos têm nota 0.** Isso não é uma nota ruim: é **ausência de nota** (normalmente filmes ainda não lançados). Trocamos a nota desses filmes por `NaN` em vez de apagar o filme, para que ele continue valendo nas contagens por ano e idioma.
3. **Notas pouco confiáveis:** uma nota calculada com 3 votos não diz muito. Criamos a coluna `nota_confiavel` (verdadeiro quando há **50 votos ou mais**) e usamos só esses filmes nas perguntas sobre nota.

In [79]:
print("Antes:", len(filmes))
filmes = filmes.dropna(subset=["popularidade", "qtd_votos", "data_lancamento"])
print("Depois de remover linhas quebradas:", len(filmes))

sem_voto = filmes["qtd_votos"] == 0
print("Filmes com 0 votos (nota vira NaN):", sem_voto.sum())
filmes.loc[sem_voto, "nota_media"] = np.nan

filmes["qtd_votos"] = filmes["qtd_votos"].astype(int)
filmes["ano"] = filmes["data_lancamento"].dt.year
filmes["decada"] = (filmes["ano"] // 10) * 10
filmes["nota_confiavel"] = filmes["qtd_votos"] >= 50
print("Filmes com nota confiável (50+ votos):", filmes["nota_confiavel"].sum())

filmes.isna().sum()

Antes: 9837
Depois de remover linhas quebradas: 9826
Filmes com 0 votos (nota vira NaN): 100
Filmes com nota confiável (50+ votos): 8796


data_lancamento      0
titulo               0
popularidade         0
qtd_votos            0
nota_media         100
idioma_original      0
generos              0
ano                  0
decada               0
nota_confiavel       0
dtype: int64

### B6. Remover duplicados
Não há linhas exatamente iguais. Há **títulos repetidos**, mas ao conferir eles são **remakes** com datas e números diferentes (ex.: *A Nightmare on Elm Street* de 1984 e de 2010). Por isso, o critério de duplicado é **título + data de lançamento**, e não só o título.

In [80]:
print("Duplicados exatos:", filmes.duplicated().sum())
print("Títulos repetidos:", filmes.duplicated(subset=["titulo"]).sum())
print("Título + data repetidos:", filmes.duplicated(subset=["titulo", "data_lancamento"]).sum())

filmes = filmes.drop_duplicates(subset=["titulo", "data_lancamento"])
filmes[filmes.duplicated(subset=["titulo"], keep=False)].sort_values("titulo")[["titulo", "ano", "qtd_votos", "nota_media"]].head(6)

Duplicados exatos: 0
Títulos repetidos: 314
Título + data repetidos: 0


,titulo,ano,qtd_votos,nota_media
3280,365 Days,2015,12,4.8
1434,365 Days,2020,6540,7.0
6003,A Christmas Carol,1984,241,7.3
2062,A Christmas Carol,2009,3732,6.9
2179,A Nightmare on Elm Street,2010,2187,5.5
1609,A Nightmare on Elm Street,1984,3877,7.3


### B7. Padronizar
- Removemos espaços extras do começo e do fim dos títulos.
- Criamos `idioma` com o nome em português dos códigos de idioma (`en` → Inglês). Os códigos sem tradução no dicionário ficam como `Outros`.
- **Gêneros:** um filme pode ter vários. Para a pergunta 5, criamos uma tabela separada (`filmes_generos.csv`) com **uma linha por filme e gênero**, usando `str.split` e `explode`. Um filme de "Ação, Aventura" conta nos dois gêneros. Também guardamos o `genero_principal` (o primeiro da lista).

In [81]:
filmes["titulo"] = filmes["titulo"].str.strip()
filmes["idioma"] = filmes["idioma_original"].map(IDIOMAS).fillna("Outros")

filmes["genero_principal"] = filmes["generos"].str.split(",").str[0].str.strip()

filmes_generos = (
    filmes.assign(genero=filmes["generos"].str.split(","))
          .explode("genero")
          .assign(genero=lambda d: d["genero"].str.strip())
          [["titulo", "ano", "decada", "genero", "qtd_votos", "nota_media", "nota_confiavel", "popularidade"]]
)
print("Filmes:", len(filmes), "| linhas filme-gênero:", len(filmes_generos))
filmes_generos["genero"].value_counts()

Filmes: 9826 | linhas filme-gênero: 25792


genero
Drama              3744
Comedy             3031
Action             2686
Thriller           2488
Adventure          1853
Romance            1476
Horror             1470
Animation          1438
Family             1414
Fantasy            1308
Science Fiction    1273
Crime              1242
Mystery             773
History             427
War                 308
Music               295
Documentary         215
TV Movie            214
Western             137
Name: count, dtype: int64

### B8. Salvar

In [82]:
filmes.to_csv(LIMPOS / "filmes_limpos.csv", index=False)
filmes_generos.to_csv(LIMPOS / "filmes_generos.csv", index=False)
filmes.head()

,data_lancamento,titulo,popularidade,qtd_votos,nota_media,idioma_original,generos,ano,decada,nota_confiavel,idioma,genero_principal
0,2021-12-15,Spider-Man: No Way Home,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",2021,2020,True,Inglês,Action
1,2022-03-01,The Batman,3827.658,1151,8.1,en,"Crime, Mystery, Thriller",2022,2020,True,Inglês,Crime
2,2022-02-25,No Exit,2618.087,122,6.3,en,Thriller,2022,2020,True,Inglês,Thriller
3,2021-11-24,Encanto,2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",2021,2020,True,Inglês,Animation
4,2021-12-22,The King's Man,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",2021,2020,True,Inglês,Action


---
## Parte C · Junção por ano

Os dois temas não têm uma chave em comum como país, então a única junção que faz sentido é pelo **ano de lançamento**. Montamos uma tabela anual com o número de lançamentos e as médias de cada tema e juntamos com `merge`.

Usamos `how="inner"` porque só interessam os anos que existem nos dois datasets. Conferimos o tamanho antes e depois, como pede o roteiro.

> Essa tabela serve para comparar tendências de lançamento lado a lado, não para afirmar que um tema causa o outro.

In [83]:
musicas_unicas = musicas.drop_duplicates(subset="id_musica")

musica_ano = musicas_unicas.groupby("ano_lancamento").agg(
    qtd_musicas=("id_musica", "count"),
    popularidade_media_musica=("popularidade_musica", "mean"),
).reset_index().rename(columns={"ano_lancamento": "ano"})

filme_ano = filmes.groupby("ano").agg(
    qtd_filmes=("titulo", "count"),
    nota_media_filme=("nota_media", "mean"),
).reset_index()

print("Anos com músicas:", len(musica_ano), "| anos com filmes:", len(filme_ano))
por_ano = musica_ano.merge(filme_ano, on="ano", how="inner")
print("Anos em comum após o merge:", len(por_ano))
print("Linhas sem par:", por_ano.isna().sum().sum())

por_ano.to_csv(LIMPOS / "musicas_filmes_por_ano.csv", index=False)
por_ano.tail()

Anos com músicas: 63 | anos com filmes: 102
Anos em comum após o merge: 63
Linhas sem par: 0


,ano,qtd_musicas,popularidade_media_musica,qtd_filmes,nota_media_filme
58,2016,1823,35.393856,470,6.366738
59,2017,2150,39.009302,510,6.422549
60,2018,2905,42.651291,530,6.497159
61,2019,7404,47.417207,500,6.686573
62,2020,626,42.781150,453,6.636062


---
## Parte D · Primeiras estatísticas

Antes dos gráficos, respondemos a cada pergunta só com números.

### Resumo numérico

In [84]:
musicas_unicas[["popularidade_musica", "energia", "fala", "acusticidade", "prob_aovivo", "duracao_min", "ano_lancamento"]].describe().round(2)

,popularidade_musica,energia,fala,acusticidade,prob_aovivo,duracao_min,ano_lancamento
count,28327.00,28327.00,28327.00,28327.00,28327.00,28327.00,28327.00
mean,39.35,0.70,0.11,0.18,0.19,3.78,2011.05
std,23.70,0.18,0.10,0.22,0.16,1.01,11.23
min,0.00,0.00,0.02,0.00,0.01,1.01,1957.00
25%,21.00,0.58,0.04,0.01,0.09,3.13,2008.00
50%,42.00,0.72,0.06,0.08,0.13,3.62,2016.00
75%,58.00,0.84,0.13,0.26,0.25,4.25,2019.00
max,100.00,1.00,0.92,0.99,1.00,8.63,2020.00


In [85]:
filmes[["popularidade", "qtd_votos", "nota_media", "ano"]].describe().round(2)

,popularidade,qtd_votos,nota_media,ano
count,9826.00,9826.00,9726.00,9826.00
mean,40.32,1392.94,6.51,2006.20
std,108.88,2611.30,0.93,15.69
min,13.35,0.00,1.50,1902.00
25%,16.13,146.00,5.90,2000.00
50%,21.20,444.00,6.50,2011.00
75%,35.18,1376.00,7.20,2017.00
max,5083.95,31077.00,10.00,2024.00


### Pergunta 1 · Quais gêneros de playlist têm as músicas mais populares?

In [86]:
p1 = (musicas.drop_duplicates(subset=["id_musica", "genero_playlist"])
             .groupby("genero_playlist")["popularidade_musica"]
             .agg(["count", "mean", "median"]).round(1)
             .sort_values("mean", ascending=False))
p1

,count,mean,median
genero_playlist,,,
Pop,5130,45.9,50.0
Latina,4624,44.2,48.0
Rap,5476,42.3,46.0
Rock,4448,40.5,45.0
R&B,5015,39.2,41.0
EDM,5436,33.6,35.0


**Resposta:** **Pop** tem a maior popularidade média (45,9), seguido de **Latina** (44,2) e **Rap** (42,3). **EDM** fica bem atrás (33,6), uma diferença de mais de 12 pontos para o Pop. A mediana segue a mesma ordem, então o resultado não vem de poucas músicas muito populares. Na Etapa 3: gráfico de **barras** ordenado.

### Pergunta 2 · A popularidade muda conforme o ano de lançamento?

In [87]:
p2 = musicas_unicas.groupby("decada")["popularidade_musica"].agg(["count", "mean"]).round(1)
print(p2)
recentes = musicas_unicas[musicas_unicas["ano_lancamento"] >= 1970]
print("\nCorrelação ano x popularidade (1970+):",
      round(np.corrcoef(recentes["ano_lancamento"], recentes["popularidade_musica"])[0, 1], 3))

        count  mean
decada             
1950        3  44.3
1960      132  46.4
1970      783  44.3
1980     1089  42.2
1990     2080  38.0
2000     3802  32.0
2010    19812  40.4
2020      626  42.8

Correlação ano x popularidade (1970+): 0.052


**Resposta:** a relação é fraca (correlação de **0,05** entre ano e popularidade a partir de 1970). O padrão é em forma de "U": músicas dos anos 60 e 70 têm média alta (46,4 e 44,3), a média cai para **32,0 nos anos 2000** e volta a subir nos anos 2010 (40,4) e 2020 (42,8). Uma hipótese é que só os clássicos antigos entram nas playlists, enquanto dos anos 2000 entra muita música menos conhecida. Atenção: 70% das músicas são dos anos 2010. Na Etapa 3: gráfico de **linha** da popularidade média por ano.

### Pergunta 3 · Características do áudio têm relação com a popularidade?

In [88]:
p3 = musicas_unicas[["popularidade_musica", "energia", "fala", "acusticidade", "prob_aovivo", "duracao_min"]].corr()
p3["popularidade_musica"].drop("popularidade_musica").round(3).sort_values()

duracao_min    -0.143
energia        -0.104
prob_aovivo    -0.052
fala            0.006
acusticidade    0.093
Name: popularidade_musica, dtype: float64

**Resposta:** nenhuma característica explica bem a popularidade, porque todas as correlações ficam entre **−0,15 e 0,10**. As mais fortes são **duração** (−0,14: músicas mais longas tendem a ser um pouco menos populares) e **energia** (−0,10). A **acusticidade** tem leve relação positiva (0,09). Na Etapa 3: **mapa de calor** de correlação (seaborn).

### Pergunta 4 · As músicas ficaram mais curtas ao longo das décadas?

In [89]:
p4 = musicas_unicas.groupby("decada")["duracao_min"].agg(["count", "mean", "median"]).round(2)
p4

,count,mean,median
decada,,,
1950,3,2.44,2.44
1960,132,3.63,3.16
1970,783,4.28,4.04
1980,1089,4.47,4.32
1990,2080,4.42,4.36
2000,3802,4.13,3.99
2010,19812,3.60,3.46
2020,626,3.24,3.14


**Resposta:** sim. A duração média subiu até os anos 80 (**4,47 min**) e desde então caiu sem parar, chegando a **3,24 min** nos anos 2020, cerca de 1 minuto e 14 segundos a menos. A mediana mostra o mesmo movimento (4,32 → 3,14). Os anos 50 têm só 3 músicas e não devem ser usados na conclusão. Na Etapa 3: **boxplot** da duração por década (seaborn).


### Pergunta 5 · Quais gêneros de filme têm as maiores notas médias?

In [90]:
p5 = (filmes_generos[filmes_generos["nota_confiavel"]]
      .groupby("genero")["nota_media"]
      .agg(["count", "mean"]).round(2)
      .sort_values("mean", ascending=False))
p5

,count,mean
genero,,
Documentary,121,7.36
History,406,7.02
Music,273,7.01
War,287,6.99
Animation,1223,6.97
Drama,3401,6.83
Western,133,6.79
Romance,1315,6.69
Family,1331,6.64


**Resposta:** entre os filmes com 50 votos ou mais, **Documentário** tem a maior nota média (**7,36**), seguido de **História** (7,02), **Música** (7,01) e **Guerra** (6,99). **Terror** tem a menor (**6,04**), atrás de Suspense (6,33) e Ficção Científica (6,38). A diferença entre o primeiro e o último gênero é de 1,3 ponto. Documentário tem só 121 filmes, então vale citar o tamanho da amostra no gráfico. Na Etapa 3: **barras horizontais**.

### Pergunta 6 · Quantidade de filmes e nota média por década

In [91]:
p6 = filmes.groupby("decada").agg(
    qtd_filmes=("titulo", "count"),
    nota_media=("nota_media", lambda s: s[filmes.loc[s.index, "nota_confiavel"]].mean()),
).round(2)
p6

,qtd_filmes,nota_media
decada,,
1900,1,8.00
1920,11,7.82
1930,33,7.41
1940,50,7.37
1950,101,7.40
1960,205,7.23
1970,269,6.94
1980,631,6.62
1990,1050,6.55


**Resposta:** a quantidade de filmes cresce muito com o tempo: são **3.998 filmes dos anos 2010**, contra 101 dos anos 50. Já a nota média **cai** de cerca de 7,4 (anos 30 a 50) para **6,42 nos anos 2000**. A explicação provável é um viés de sobrevivência: dos filmes antigos, só os clássicos continuam populares no TMDB. Os anos 1900 e 1920 têm poucos filmes (1 e 11) e não devem pesar na conclusão. Na Etapa 3: **barras** (quantidade) e **linha** (nota) no mesmo gráfico.

### Pergunta 7 · Filmes com mais votos têm notas mais altas?

In [92]:
conf = filmes[filmes["nota_confiavel"]]
r_bruto = np.corrcoef(conf["qtd_votos"], conf["nota_media"])[0, 1]
r_log = np.corrcoef(np.log10(conf["qtd_votos"]), conf["nota_media"])[0, 1]
print("Correlação votos x nota:", round(r_bruto, 3))
print("Correlação log10(votos) x nota:", round(r_log, 3))

conf.assign(faixa_votos=pd.qcut(conf["qtd_votos"], 4, labels=["1º quartil", "2º", "3º", "4º quartil"])) \
    .groupby("faixa_votos", observed=True)["nota_media"].mean().round(2)

Correlação votos x nota: 0.278
Correlação log10(votos) x nota: 0.253


faixa_votos
1º quartil    6.37
2º            6.44
3º            6.54
4º quartil    6.89
Name: nota_media, dtype: float64

**Resposta:** sim, mas de forma moderada a fraca. A correlação entre votos e nota é de **0,28**. Dividindo os filmes em quatro faixas de votos, a nota média sobe de **6,37** (25% com menos votos) para **6,89** (25% com mais votos). Filmes mais vistos tendem a ser um pouco mais bem avaliados, mas há muitos filmes com nota alta e poucos votos. Na Etapa 3: gráfico de **dispersão** com votos em escala logarítmica.

### Pergunta 8 · Quais idiomas originais aparecem mais e como são avaliados?

In [93]:
p8 = (filmes.groupby("idioma")
      .agg(qtd_filmes=("titulo", "count"),
           pct=("titulo", lambda s: 100 * len(s) / len(filmes)),
           nota_media=("nota_media", lambda s: s[filmes.loc[s.index, "nota_confiavel"]].mean()))
      .round(2).sort_values("qtd_filmes", ascending=False))
p8.head(8)

,qtd_filmes,pct,nota_media
idioma,,,
Inglês,7569,77.03,6.46
Japonês,645,6.56,7.19
Espanhol,339,3.45,6.99
Francês,292,2.97,6.63
Coreano,170,1.73,7.26
Chinês,129,1.31,6.85
Italiano,123,1.25,6.71
Cantonês,109,1.11,6.85


**Resposta:** o dataset é dominado pelo **inglês** (**77%** dos filmes), seguido de **japonês** (6,6%) e **espanhol** (3,5%). Os filmes em inglês têm a **menor** nota média entre os idiomas principais (**6,46**), enquanto **coreano** (7,26) e **japonês** (7,19) têm as maiores. Provavelmente só os filmes estrangeiros de maior destaque chegam a ficar populares no TMDB. Na Etapa 3: **barras** de quantidade com a nota média anotada.

###  · Junção por ano

In [94]:
recorte = por_ano[por_ano["ano"] >= 1980]
print("Correlação qtd de músicas x qtd de filmes por ano (1980+):",
      round(np.corrcoef(recorte["qtd_musicas"], recorte["qtd_filmes"])[0, 1], 3))

Correlação qtd de músicas x qtd de filmes por ano (1980+): 0.697


**Resposta:** a quantidade de músicas e de filmes por ano anda junto (correlação de **0,70** desde 1980), mas isso reflete principalmente que os **dois datasets têm mais itens recentes**, e não uma relação entre música e cinema. Usaremos essa tabela só como contexto.

---
### Próximo passo (Etapa 3)
Os arquivos de `data/limpos/` alimentam o notebook `02_graficos.ipynb`, que desenha um gráfico para cada pergunta e salva tudo em `images/`.